# ADEGI — Fine-tuning de YOLOv8n (7 clases) a 256×256 + export TFLite int8

Clases: `person, chair, couch, bed, dining table, cane, walker`. Ver `docs/vision-research.md` y `vision/README.md`.

**Por qué este flujo (en una línea por paso)**
1. *Auto-etiquetar con yolov8x*: las imágenes de bastón/andadera solo traen esas dos clases etiquetadas; sin ayuda, el modelo aprendería que sillas, camas y personas son fondo (olvido catastrófico).
2. *Fusionar + remapear por nombre*: cada dataset numera distinto sus clases.
3. *imgsz=256*: es la resolución del benchmark de ST para la NPU del MP257; entrenar a la resolución de despliegue evita perder precisión por reescalar.
4. *Exportar int8 con calibración real*: la cuantización estática necesita imágenes representativas para fijar los rangos.

Runtime → GPU (Entorno de ejecución ▸ Cambiar tipo ▸ T4).

In [ ]:
!pip install -q ultralytics pyyaml
import ultralytics; ultralytics.checks()
print("ultralytics", ultralytics.__version__)  # anota esta versión: los flags de export cambian entre versiones

## 1. Código del repo
Si el repo es privado, usa un token (`https://<TOKEN>@github.com/...`) o sube la carpeta `vision/` a Drive y monta Drive.

In [ ]:
REPO_URL = "https://github.com/etherial-art/adegi.git"
BRANCH = "main"   # cámbialo a la rama que quieras probar
!git clone --depth 1 -b $BRANCH $REPO_URL /content/adegi
%cd /content/adegi/vision

## 2. Datos
Coloca los datasets ya descargados en estas rutas (formato YOLO: `train/images`, `train/labels`, `data.yaml`).

- **Bastón/andadera**: Roboflow Universe (búsqueda "walker cane") o Open Images V7. **Revisa y anota la licencia de cada uno**; ADEGI es producto.
- Las rutas y licencias de abajo son *placeholders*: edítalas.

In [ ]:
from pathlib import Path
CANEWALKER = Path("/content/data/canewalker")      # dataset(s) con bastón/andadera
CANEWALKER_LICENSE = "CHECK-ME"                    # p. ej. "CC BY 4.0"
COCO_SUBSET = Path("/content/data/coco_subset")    # opcional: camas/sillas/etc. para balancear (sección 3)
COCO_LICENSE = "CC BY 4.0 (anotaciones COCO; imágenes: licencias Flickr individuales, revisar)"
OUT = Path("/content/adegi_det")

## 3. (Opcional) Subconjunto de COCO con las clases de interés
Equilibra el dataset: sin esto casi todas las imágenes serían de bastón/andadera.

> **No verificado:** el uso de `fiftyone` (nombres de argumentos del zoo / export) está escrito de memoria. Si falla, descarga los subconjuntos de COCO a mano y expórtalos en formato YOLO.

In [ ]:
# !pip install -q fiftyone
# import fiftyone as fo, fiftyone.zoo as foz
# ds = foz.load_zoo_dataset("coco-2017", split="train", label_types=["detections"],
#                           classes=["person", "chair", "couch", "bed", "dining table"], max_samples=2000)
# ds.export(export_dir=str(COCO_SUBSET), dataset_type=fo.types.YOLOv5Dataset, label_field="ground_truth",
#           split="train")

## 4. Auto-etiquetado con yolov8x
Solo para el dataset de bastón/andadera (el subconjunto de COCO ya viene etiquetado por humanos).
Filtra `classes=[0, 56, 57, 59, 60]` y usa `conf` alto: una etiqueta falsa daña más que una faltante.

In [ ]:
import glob
IMG_DIRS = glob.glob(str(CANEWALKER / "*" / "images"))
print("image folders:", IMG_DIRS)
!python -m adegi_vision.tools.autolabel --images {CANEWALKER} --out /content/autolabels --model yolov8x.pt --conf 0.45

## 5. Fusión + `data.yaml`
Remapea por nombre al esquema de 7 clases, conserva los splits, añade las auto-etiquetas (las humanas ganan en duplicados) y escribe `dataset_manifest.json` con licencias.

In [ ]:
sources = f'--source canewalker {CANEWALKER} "{CANEWALKER_LICENSE}"'
if COCO_SUBSET.exists():
    sources += f' --source coco_subset {COCO_SUBSET} "{COCO_LICENSE}"'
!python -m adegi_vision.tools.merge_dataset --out {OUT} --autolabels /content/autolabels {sources}
!cat {OUT}/data.yaml

## 6. Revisión manual (obligatoria)
Mira una muestra: el auto-etiquetado se equivoca. Corrige en CVAT/Roboflow lo que veas mal y vuelve a la sección 5 si hace falta.

In [ ]:
import random, cv2, matplotlib.pyplot as plt
from adegi_vision.classes import CLASS_NAMES
imgs = sorted((OUT / "images" / "train").glob("*"))
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, p in zip(axes.ravel(), random.sample(imgs, min(6, len(imgs)))):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB); H, W = im.shape[:2]
    for line in (OUT / "labels" / "train" / f"{p.stem}.txt").read_text().splitlines():
        c, xc, yc, w, h = line.split(); c = int(c); xc, yc, w, h = [float(v) for v in (xc, yc, w, h)]
        x1, y1, x2, y2 = int((xc-w/2)*W), int((yc-h/2)*H), int((xc+w/2)*W), int((yc+h/2)*H)
        cv2.rectangle(im, (x1, y1), (x2, y2), (255, 0, 0), 3); cv2.putText(im, CLASS_NAMES[c], (x1, y1+25), 0, 1, (255, 255, 0), 2)
    ax.imshow(im); ax.axis("off")
plt.show()

## 7. Entrenamiento
`yolov8n.pt` preentrenado (COCO) → fine-tuning a 7 clases. Cumple el requisito de la materia (CNN preentrenada con fine-tuning).
Aumentos por defecto de Ultralytics (mosaic, flip, HSV) ya ayudan; `patience` detiene si no mejora.

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8n.pt")
results = model.train(data=str(OUT / "data.yaml"), imgsz=256, epochs=100, batch=64, patience=20,
                      project="/content/runs", name="adegi_det256", seed=0)
best = Path(results.save_dir) / "weights" / "best.pt"
print(best)

In [ ]:
# mAP por clase (float). Fíjate sobre todo en cane y walker: son las clases nuevas.
m = YOLO(str(best)).val(data=str(OUT / "data.yaml"), imgsz=256)
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:14s} mAP50-95 = {m.box.maps[i]:.3f}")
print("mAP50-95 global =", round(m.box.map, 3))

## 8. Export TFLite int8 (cuantización estática)
Verificado en la doc de Ultralytics (`modes/export.md`, rama main): existe el argumento `quantize` (`8`, `"int8"`, ...) y `int8=True` es un alias *legado*; `data=` da las imágenes de calibración.
El resto depende de **tu versión** instalada (impresa arriba), por eso se prueba `quantize=8` y se cae a `int8=True`.

> ⚠️ **Per-tensor vs per-channel (NO verificado en Ultralytics):** la doc de Ultralytics no dice qué granularidad produce. Según la wiki de ST, la NPU del MP25 necesita **per-tensor asimétrico** y un modelo per-channel corre sobre todo en GPU. Asume que este `.tflite` es per-channel hasta inspeccionarlo (Netron / `interpreter.get_tensor_details()` → `quantization_parameters['scales']`: un solo valor = per-tensor, uno por canal = per-channel). Para la NPU, cuantiza con los scripts de `stm32ai-modelzoo-services` (ver `vision/README.md`).

In [ ]:
best_model = YOLO(str(best))
try:
    exported = best_model.export(format="tflite", imgsz=256, quantize=8, data=str(OUT / "data.yaml"))
except (SyntaxError, TypeError, KeyError) as e:   # versiones viejas no conocen `quantize`
    print("quantize= not accepted by this ultralytics version -> falling back to int8=True:", e)
    exported = best_model.export(format="tflite", imgsz=256, int8=True, data=str(OUT / "data.yaml"))
print("export returned:", exported)
tfl = sorted(Path(best).parent.rglob("*.tflite"))
print("\n".join(map(str, tfl)))

In [ ]:
# Calidad antes/después de cuantizar. Si mAP cae > ~3 puntos: más imágenes de calibración o revisar el dataset.
int8_files = [p for p in tfl if "int8" in p.name]
if int8_files:
    m8 = YOLO(str(int8_files[0]), task="detect").val(data=str(OUT / "data.yaml"), imgsz=256)   # no verificado en todas las versiones
    print("float mAP50-95:", round(m.box.map, 3), " int8 mAP50-95:", round(m8.box.map, 3))
else:
    print("no *int8*.tflite found; inspect the list above")

In [ ]:
# Inspección de la granularidad de cuantización del .tflite (1 escala = per-tensor; N escalas = per-channel)
import numpy as np, tensorflow as tf
if int8_files:
    it = tf.lite.Interpreter(model_path=str(int8_files[0])); it.allocate_tensors()
    per_channel = sum(1 for d in it.get_tensor_details() if len(d["quantization_parameters"]["scales"]) > 1)
    print("tensors with per-channel scales:", per_channel)

In [ ]:
# Guarda resultados en Drive (el disco de Colab se borra)
from google.colab import drive; drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/adegi && cp {best} /content/drive/MyDrive/adegi/adegi_det256_best.pt
!cp {OUT}/dataset_manifest.json /content/drive/MyDrive/adegi/
for p in tfl: !cp "{p}" /content/drive/MyDrive/adegi/